# 03 — Poz Modeli Karsilastirmasi (Katman 2, G2)
MediaPipe Pose ve RTMPose ayni goruntuler uzerinde dogruluk, cikarim hizi ve guven dagilimi
acisindan karsilastirilir. Model degistirmeye olanak taniyan adaptör arayuzu dogrulanir.

## Mimari Sozlesme ve Arastirma Ilkeleri

Bu notebook, **PhysioLoad AI — Olcum Duyarli Secici Biyomekanik Analiz Mimarisi**'nin bir parcasidir:
1. **Once olc, sonra guvenilirligini sinayarak raporla:** Her biyomekanik cikti bagimsiz degerlendirilir.
2. **Uc durumlu karar sozlesmesi:** `ACCEPTED` (Kabul), `REJECTED` (Ret), `NOT_COMPUTABLE` (Hesaplanamaz).
3. **Ayri dogrulama hatti:** Referans eklem verileri asla model tahminine girdi yapilmaz; yalnizca hata analizi ve A0-A4 karsilastirmasinda kullanilir.
4. **Veri sizintisini onleme:** Kisi ve deneme bazli boluntuleme (ayni katilimci train/val/test arasinda karismaz).
5. **Esik secimi:** Sayisal esikler yalnizca validation bolumunde belirlenir; test kumesi karsilastirma icin saklanir.


In [1]:
import sys
from pathlib import Path

# Proje kok dizini ve ortam kurulumu
here = Path.cwd().resolve()
code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))

from physioload_env import setup_environment
env = setup_environment(root=code_root)
print("Ortam Basariyla Yuklendi:")
print(env.summary())


Ortam Basariyla Yuklendi:
context       : vscode local
user          : berk-can
processed ver : v1
device        : cpu
code root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
artifacts root: C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
data root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai


In [2]:
import cv2
import numpy as np
import pandas as pd
from physioload.pose import MediaPipePoseEstimator, RTMPoseEstimator

# Test icin sentetik insan silueti / manken olustur
synthetic_canvas = np.zeros((480, 640, 3), dtype=np.uint8)
cv2.circle(synthetic_canvas, (320, 90), 28, (220, 220, 220), -1)   # Bas
cv2.line(synthetic_canvas, (320, 118), (320, 270), (220, 220, 220), 6) # Govde
cv2.line(synthetic_canvas, (320, 150), (250, 210), (220, 220, 220), 5) # Sol Kol
cv2.line(synthetic_canvas, (320, 150), (390, 210), (220, 220, 220), 5) # Sag Kol
cv2.line(synthetic_canvas, (320, 270), (280, 410), (220, 220, 220), 6) # Sol Bacak
cv2.line(synthetic_canvas, (320, 270), (360, 410), (220, 220, 220), 6) # Sag Bacak

# Modellerin hazirlanmasi
estimators = {}
for name, cls in [('mediapipe', MediaPipePoseEstimator), ('rtmpose', RTMPoseEstimator)]:
    try:
        estimators[name] = cls()
        print(f"Modul basariyla baslatildi: {name}")
    except Exception as e:
        print(f"{name} baslatilamadi: {e}")


Modul basariyla baslatildi: mediapipe
load C:\Users\Berk\.cache\rtmlib\hub\checkpoints\yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend


load C:\Users\Berk\.cache\rtmlib\hub\checkpoints\rtmpose-s_simcc-body7_pt-body7_420e-256x192-acd4a1ef_20230504.onnx with onnxruntime backend
Modul basariyla baslatildi: rtmpose


In [3]:
# Ortak Karsilastirma Protokolu (G2)
benchmark_results = []
for name, model in estimators.items():
    try:
        # Cikarim yap
        res = model.predict(synthetic_canvas)
        benchmark_results.append({
            'Model': name,
            'Landmark Sayisi': len(res.names),
            'Ortalama Guven': float(np.mean(res.confidence)),
            'Cikarim Suresi (ms)': round(res.inference_ms, 2),
            'Backend': res.backend,
        })
    except Exception as err:
        print(f"{name} cikarim sirasinda hata: {err}")

df_benchmark = pd.DataFrame(benchmark_results)
print("\nModel Karsilastirma Tablosu (G2):")
print(df_benchmark.to_string(index=False))


mediapipe cikarim sirasinda hata: MediaPipe did not detect a pose

Model Karsilastirma Tablosu (G2):
  Model  Landmark Sayisi  Ortalama Guven  Cikarim Suresi (ms) Backend
rtmpose               17        0.191665                58.01 rtmpose


Ortak landmark eşlemesi (`physioload_17`) olmadan koordinatlar doğrudan karşılaştırılamaz. MediaPipe görünürlük ile RTMPose güveni model bazında kalibre edilir.